In [ ]:
from optics_design_workbench.jupyter_utils import *
import sympy as sy
from matplotlib.pyplot import *
from numpy import *

# Run tests with zero focal length

In [ ]:
results = []
with FreecadDocument( workInTempCopy=True ) as f:
  for distribution in (
    'exp(-theta**2/0.01**2)',
    'exp(-theta**2/0.03**2)',
    '1',
    'cos(30*theta)**2',
    '2-abs(theta)',
  ):
    for thetaDomain in (
      '0, .1',
      '-.1, .1',
      '-.02, -.01',
    ):
      # update simulation settings
      f.OpticalSimulationSettings.EndAfterRays = 'inf'
      f.OpticalSimulationSettings.EndAfterHits = '1e5'

      # update source parameters
      s = f.OpticalPointSource
      s.FocalLength = '0'
      s.PowerDensity = distribution
      s.ThetaDomain = thetaDomain
      s.PhiDomain = '0, 2*pi'

      # run fan simulation
      results.append([ s.PowerDensity.get(), 
                       s.ThetaDomain.get(),
                       f.runSimulation('true') ])

### Calc and plot diffs between simulation and expectation

In [ ]:
rmsErrs = []
for (dens, domain, r) in results:
  _, (ax1, ax2, ax3, ax4) = subplots(1,4,figsize=(15,4))

  # calc and plot cartesian mode histogram
  sca(ax1)
  H = r.loadHits('*')
  hist = H.histogram(bins=30)
  hist.plot()

  # calc cartesian histogram rms err
  X, Y = meshgrid((hist.binX[1:]+hist.binX[:-1])/2, (hist.binY[1:]+hist.binY[:-1])/2)
  expect = sy.lambdify('theta', dens)(atan(sqrt(X**2+Y**2)/100))
  if not hasattr(expect, '__len__'):
    expect = array([expect]*len(X))
  scaledRms = lambda a: sqrt(mean((a*hist.hist-expect)**2))/max(expect)
  a = scipy.optimize.minimize_scalar(scaledRms).x
  rmsErr = scaledRms(a)
  rmsErrs.append(rmsErr)
  sca(ax2)
  pcolormesh(X, Y, expect-a*hist.hist)
  colorbar()
  title(f'{rmsErr=:.0e}')

  # calc and plot polar mode histogram
  sca(ax3)
  hist = H.histogram(bins=(3,50), binCoords='polar')
  hist.plot()

  # calc polar histogram rms err
  sca(ax4)
  phis, rads, A = hist.byAzimuth()
  A = [_A[abs(rads)<5] for _A in A]
  rads = rads[abs(rads)<5]
  expect = sy.lambdify('theta', dens)(atan(rads/100))
  if not hasattr(expect, '__len__'):
    expect = array([expect]*len(rads))
  scaledRms = lambda a: sqrt(mean([mean((a*_A-expect)**2) for _A in A]))/max(expect)
  a = scipy.optimize.minimize_scalar(scaledRms).x
  rmsErr = scaledRms(a)
  rmsErrs.append(rmsErr)
  for _A in A:
    plot(rads, a*_A)
  plot(rads, expect, lw=5, alpha=.5, label=f'{rmsErr=:.0e}')
  legend()
  title('expected and simulated radial')
  suptitle(f'{dens}, {domain}')

In [ ]:
median(rmsErrs), max(rmsErrs)

In [ ]:
assert median(rmsErrs) < 0.3

In [ ]:
assert max(rmsErrs) < 3

# Run tests with infinite focal length

In [ ]:
#openFreecadGui()

In [ ]:
results = []
with FreecadDocument( workInTempCopy=False ) as f:
  for distribution in (
    'exp(-r**2/1**2)',
    'exp(-r**2/3**2)',
    '1',
    'cos(r/3)**2',
    '10-abs(r)',
  ):
    for radiusDomain in (
      '0, 10',
      '-10, 10',
      '-2, -1',
    ):
      # update simulation settings
      f.OpticalSimulationSettings.EndAfterRays = 'inf'
      f.OpticalSimulationSettings.EndAfterHits = '1e5'

      # update source parameters
      s = f.OpticalPointSource
      s.PowerDensity = distribution
      s.FocalLength = 'inf'
      s.RadiusDomain = radiusDomain
      s.PhiDomain = '0, 2*pi'

      # run fan simulation
      results.append([ s.PowerDensity.get(), 
                       s.RadiusDomain.get(),
                       f.runSimulation('true') ])

In [ ]:
rmsErrs = []
for (dens, domain, r) in results:
  _, (ax1, ax2, ax3, ax4) = subplots(1,4,figsize=(15,4))

  # calc and plot cartesian mode histogram
  sca(ax1)
  H = r.loadHits('*')
  hist = H.histogram(bins=30)
  hist.plot()

  # calc cartesian histogram rms err
  X, Y = meshgrid((hist.binX[1:]+hist.binX[:-1])/2, (hist.binY[1:]+hist.binY[:-1])/2)
  expect = sy.lambdify('r', dens)(sqrt(X**2+Y**2))
  scaledRms = lambda a: sqrt(mean((a*hist.hist-expect)**2))/max(expect)
  if not hasattr(expect, '__len__'):
    expect = array([expect]*len(X))
  a = scipy.optimize.minimize_scalar(scaledRms).x
  rmsErr = scaledRms(a)
  rmsErrs.append(rmsErr)
  sca(ax2)
  pcolormesh(X, Y, expect-a*hist.hist)
  colorbar()
  title(f'{rmsErr=:.0e}')

  # calc and plot polar mode histogram
  sca(ax3)
  hist = H.histogram(bins=(3,50), binCoords='polar')
  hist.plot()

  # calc polar histogram rms err
  sca(ax4)
  phis, rads, A = hist.byAzimuth()
  A = [_A[abs(rads)<5] for _A in A]
  rads = rads[abs(rads)<5]
  expect = sy.lambdify('r', dens)(rads)
  if not hasattr(expect, '__len__'):
    expect = array([expect]*len(rads))
  scaledRms = lambda a: sqrt(mean([mean((a*_A-expect)**2) for _A in A]))/max(expect)
  a = scipy.optimize.minimize_scalar(scaledRms).x
  rmsErr = scaledRms(a)
  rmsErrs.append(rmsErr)
  for _A in A:
    plot(rads, a*_A)
  plot(rads, expect, lw=5, alpha=.5, label=f'{rmsErr=:.0e}')
  title('expected and simulated radial')
  legend()

In [ ]:
median(rmsErrs), max(rmsErrs)

In [ ]:
assert median(rmsErrs) < 0.3

In [ ]:
assert max(rmsErrs) < 1.5